In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (6082007, 11)
Validation: (191147, 11)
Test: (89466, 11)


In [2]:
TARGET = "isFraud"
ID_COLUMNS = ["nameOrig", "nameDest"]

X_train = train_df.drop(columns=[TARGET] + ID_COLUMNS)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET] + ID_COLUMNS)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + ID_COLUMNS)
y_test = test_df[TARGET]

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = ["type"]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [4]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [5]:
val_prob = logistic_model.predict_proba(X_val)[:, 1]
test_prob = logistic_model.predict_proba(X_test)[:, 1]

print(val_prob[:10])

[0.95786185 0.76270916 1.         1.         0.93053632 0.64657652
 0.97720079 0.85374938 0.99954724 0.99581899]


In [6]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val, val_pred))

print("ROC-AUC:",
      roc_auc_score(y_val, val_prob))

print("PR-AUC:",
      average_precision_score(y_val, val_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_val, val_pred))

              precision    recall  f1-score   support

           0       1.00      0.93      0.97    189967
           1       0.09      1.00      0.16      1180

    accuracy                           0.94    191147
   macro avg       0.54      0.97      0.56    191147
weighted avg       0.99      0.94      0.96    191147

ROC-AUC: 0.993999662564051
PR-AUC: 0.7538381568128828
Confusion Matrix:
[[177548  12419]
 [     5   1175]]


In [7]:
val_pred = (val_prob >= 0.5).astype(int)

In [8]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(0.01, 1.00, 0.01):

    pred = (val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1
97,0.98,0.752049,0.622034,0.680891
98,0.99,0.840652,0.567797,0.677795
96,0.97,0.690476,0.663559,0.676750
95,0.96,0.637285,0.692373,0.663688
94,0.95,0.597742,0.717797,0.652291
93,0.94,0.561133,0.738983,0.637893
92,0.93,0.525641,0.764407,0.622928
91,0.92,0.490374,0.777119,0.601311
90,0.91,0.458738,0.800847,0.583333
89,0.90,0.428128,0.817797,0.562027


In [9]:
threshold_df[
    threshold_df["recall"] >= 0.90
].sort_values(
    "precision",
    ascending=False
).head(10)

,threshold,precision,recall,f1
74,0.75,0.211321,0.901695,0.342397
73,0.74,0.204833,0.905085,0.334063
72,0.73,0.197715,0.909322,0.324807
71,0.72,0.190890,0.912712,0.315743
70,0.71,0.184705,0.916949,0.307474
69,0.70,0.178166,0.916949,0.298359
68,0.69,0.171868,0.919492,0.289604
67,0.68,0.166413,0.925424,0.282098
66,0.67,0.161271,0.928814,0.274824
65,0.66,0.156024,0.933898,0.267378


In [10]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

TARGET = "isFraud"
ID_COLUMNS = ["nameOrig", "nameDest"]

X_train = train_df.drop(columns=[TARGET] + ID_COLUMNS)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET] + ID_COLUMNS)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + ID_COLUMNS)
y_test = test_df[TARGET]

In [11]:
X_train = pd.get_dummies(
    X_train,
    columns=["type"],
    dtype=int
)

X_val = pd.get_dummies(
    X_val,
    columns=["type"],
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=["type"],
    dtype=int
)

X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

In [14]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

rf_val_pred = (rf_val_prob >= 0.5).astype(int)

print(classification_report(
    y_val,
    rf_val_pred,
    zero_division=0
))

print("ROC-AUC:",
      roc_auc_score(y_val, rf_val_prob))

print("PR-AUC:",
      average_precision_score(y_val, rf_val_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    189967
           1       1.00      0.74      0.85      1180

    accuracy                           1.00    191147
   macro avg       1.00      0.87      0.93    191147
weighted avg       1.00      1.00      1.00    191147

ROC-AUC: 0.9997043018979301
PR-AUC: 0.9670073350133593

Confusion Matrix:
[[189966      1]
 [   303    877]]


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score

rf_threshold_results = []

for threshold in np.arange(0.01, 1.00, 0.01):

    pred = (rf_val_prob >= threshold).astype(int)

    rf_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        )
    })

rf_threshold_df = pd.DataFrame(rf_threshold_results)

rf_threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1
18,0.19,0.925158,0.869492,0.896461
19,0.20,0.935065,0.854237,0.892826
20,0.21,0.947619,0.843220,0.892377
16,0.17,0.889545,0.894068,0.891801
17,0.18,0.901995,0.881356,0.891556
21,0.22,0.957031,0.830508,0.889292
22,0.23,0.964286,0.823729,0.888483
15,0.16,0.867802,0.906780,0.886863
23,0.24,0.967742,0.813559,0.883978
24,0.25,0.975385,0.805932,0.882599


In [16]:
rf_threshold_df[
    rf_threshold_df["recall"] >= 0.90
].sort_values(
    "precision",
    ascending=False
).head(10)

,threshold,precision,recall,f1
15,0.16,0.867802,0.906780,0.886863
14,0.15,0.845914,0.921186,0.881947
13,0.14,0.825468,0.933898,0.876342
12,0.13,0.793030,0.944915,0.862336
11,0.12,0.753498,0.958475,0.843715
10,0.11,0.715003,0.965254,0.821493
9,0.10,0.674897,0.972881,0.796946
8,0.09,0.624056,0.980508,0.762690
7,0.08,0.575518,0.988136,0.727386
6,0.07,0.522940,0.994915,0.685547


In [17]:
comparison = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "roc_auc": roc_auc_score(y_val, val_prob),
        "pr_auc": average_precision_score(y_val, val_prob)
    },
    {
        "model": "Random Forest",
        "roc_auc": roc_auc_score(y_val, rf_val_prob),
        "pr_auc": average_precision_score(y_val, rf_val_prob)
    }
])

comparison

,model,roc_auc,pr_auc
0,Logistic Regression,0.994000,0.753838
1,Random Forest,0.999704,0.967007


In [18]:
from xgboost import XGBClassifier

In [19]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative:", negative)
print("Positive:", positive)
print("scale_pos_weight:", scale_pos_weight)

Negative: 6076226
Positive: 5781
scale_pos_weight: 1051.0683272790175


In [20]:
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'aucpr'


In [21]:
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

In [22]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

xgb_roc_auc = roc_auc_score(
    y_val,
    xgb_val_prob
)

xgb_pr_auc = average_precision_score(
    y_val,
    xgb_val_prob
)

print("XGBoost ROC-AUC:", xgb_roc_auc)
print("XGBoost PR-AUC:", xgb_pr_auc)

XGBoost ROC-AUC: 0.9999134550844826
XGBoost PR-AUC: 0.9874903275259413


In [23]:
comparison = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "roc_auc": roc_auc_score(y_val, val_prob),
        "pr_auc": average_precision_score(y_val, val_prob)
    },
    {
        "model": "Random Forest",
        "roc_auc": roc_auc_score(y_val, rf_val_prob),
        "pr_auc": average_precision_score(y_val, rf_val_prob)
    },
    {
        "model": "XGBoost",
        "roc_auc": xgb_roc_auc,
        "pr_auc": xgb_pr_auc
    }
])

comparison.sort_values(
    "pr_auc",
    ascending=False
)

,model,roc_auc,pr_auc
2,XGBoost,0.999913,0.987490
1,Random Forest,0.999704,0.967007
0,Logistic Regression,0.994000,0.753838


In [24]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

xgb_threshold_results = []

for threshold in np.arange(0.01, 1.00, 0.01):

    pred = (xgb_val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        pred
    ).ravel()

    xgb_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "false_negatives": fn
    })

xgb_threshold_df = pd.DataFrame(
    xgb_threshold_results
)

In [25]:
xgb_threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1,false_positives,false_negatives
98,0.99,0.917609,0.962712,0.939620,102,44
97,0.98,0.880212,0.983898,0.929172,158,19
96,0.97,0.851933,0.989831,0.915719,203,12
95,0.96,0.827562,0.992373,0.902505,244,9
94,0.95,0.807851,0.994068,0.891337,279,7
93,0.94,0.793103,0.994068,0.882287,306,7
92,0.93,0.780212,0.995763,0.874907,331,5
91,0.92,0.765472,0.995763,0.865562,360,5
90,0.91,0.750958,0.996610,0.856519,390,4
89,0.90,0.738095,0.998305,0.848703,418,2


In [26]:
xgb_threshold_df[
    xgb_threshold_df["recall"] >= 0.95
].sort_values(
    "precision",
    ascending=False
).head(10)

,threshold,precision,recall,f1,false_positives,false_negatives
98,0.99,0.917609,0.962712,0.939620,102,44
97,0.98,0.880212,0.983898,0.929172,158,19
96,0.97,0.851933,0.989831,0.915719,203,12
95,0.96,0.827562,0.992373,0.902505,244,9
94,0.95,0.807851,0.994068,0.891337,279,7
93,0.94,0.793103,0.994068,0.882287,306,7
92,0.93,0.780212,0.995763,0.874907,331,5
91,0.92,0.765472,0.995763,0.865562,360,5
90,0.91,0.750958,0.996610,0.856519,390,4
89,0.90,0.738095,0.998305,0.848703,418,2


In [27]:
xgb_threshold_df[
    xgb_threshold_df["recall"] >= 0.99
].sort_values(
    "precision",
    ascending=False
).head(10)

,threshold,precision,recall,f1,false_positives,false_negatives
95,0.96,0.827562,0.992373,0.902505,244,9
94,0.95,0.807851,0.994068,0.891337,279,7
93,0.94,0.793103,0.994068,0.882287,306,7
92,0.93,0.780212,0.995763,0.874907,331,5
91,0.92,0.765472,0.995763,0.865562,360,5
90,0.91,0.750958,0.996610,0.856519,390,4
89,0.90,0.738095,0.998305,0.848703,418,2
88,0.89,0.731223,0.998305,0.844142,433,2
87,0.88,0.720049,0.998305,0.836648,458,2
86,0.87,0.711782,0.998305,0.831041,477,2


In [28]:
best_threshold_row = (
    xgb_threshold_df[
        xgb_threshold_df["recall"] >= 0.95
    ]
    .sort_values(
        ["precision", "f1"],
        ascending=False
    )
    .iloc[0]
)

best_threshold_row

threshold            0.990000
precision            0.917609
recall               0.962712
f1                   0.939620
false_positives    102.000000
false_negatives     44.000000
Name: 98, dtype: float64

In [29]:
BEST_THRESHOLD = best_threshold_row["threshold"]

print("Selected threshold:", BEST_THRESHOLD)
print("Validation precision:", best_threshold_row["precision"])
print("Validation recall:", best_threshold_row["recall"])
print("Validation F1:", best_threshold_row["f1"])

Selected threshold: 0.99
Validation precision: 0.9176090468497576
Validation recall: 0.9627118644067797
Validation F1: 0.9396195202646815


In [30]:
xgb_test_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

In [31]:
xgb_test_pred = (
    xgb_test_prob >= BEST_THRESHOLD
).astype(int)

In [32]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print(
    classification_report(
        y_test,
        xgb_test_pred,
        digits=4,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, xgb_test_prob)
)

print(
    "PR-AUC:",
    average_precision_score(y_test, xgb_test_prob)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_test_pred))

              precision    recall  f1-score   support

           0     0.9993    0.9993    0.9993     88214
           1     0.9490    0.9521    0.9506      1252

    accuracy                         0.9986     89466
   macro avg     0.9742    0.9757    0.9749     89466
weighted avg     0.9986    0.9986    0.9986     89466

ROC-AUC: 0.9998578102003037
PR-AUC: 0.9901719774016853

Confusion Matrix:
[[88150    64]
 [   60  1192]]


In [33]:
tn, fp, fn, tp = confusion_matrix(
    y_test,
    xgb_test_pred
).ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn)

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Positive Rate: {false_positive_rate:.6f}")
print(f"False Negative Rate: {false_negative_rate:.6f}")

True Negatives : 88150
False Positives: 64
False Negatives: 60
True Positives : 1192

Precision: 0.9490
Recall: 0.9521
False Positive Rate: 0.000726
False Negative Rate: 0.047923


In [34]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

test_roc_auc = roc_auc_score(
    y_test,
    xgb_test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    xgb_test_prob
)

print(f"Test ROC-AUC: {test_roc_auc:.6f}")
print(f"Test PR-AUC:  {test_pr_auc:.6f}")

Test ROC-AUC: 0.999858
Test PR-AUC:  0.990172


In [35]:
import joblib

joblib.dump(
    xgb_model,
    "../data/processed/xgb_fraud_model.pkl"
)

print("XGBoost model saved.")

XGBoost model saved.


In [36]:
import json

with open(
    "../data/processed/fraud_threshold.json",
    "w"
) as f:
    json.dump(
        {
            "threshold": float(BEST_THRESHOLD)
        },
        f,
        indent=4
    )

print("Threshold saved:", BEST_THRESHOLD)

Threshold saved: 0.99


In [39]:
from src.features.engineering import create_features
from src.features.preprocessing import create_preprocessor

In [38]:
import sys
sys.path.append("..")

In [40]:
train_raw = pd.read_csv(
    "../data/processed/train.csv"
)

val_raw = pd.read_csv(
    "../data/processed/validation.csv"
)

test_raw = pd.read_csv(
    "../data/processed/test.csv"
)

In [41]:
y_train = train_raw["isFraud"]
y_val = val_raw["isFraud"]
y_test = test_raw["isFraud"]

X_train_raw = train_raw.drop(columns=["isFraud"])
X_val_raw = val_raw.drop(columns=["isFraud"])
X_test_raw = test_raw.drop(columns=["isFraud"])

In [42]:
X_train_features = create_features(X_train_raw)
X_val_features = create_features(X_val_raw)
X_test_features = create_features(X_test_raw)

In [43]:
preprocessor = create_preprocessor(
    X_train_features
)

X_train = preprocessor.fit_transform(
    X_train_features
)

X_val = preprocessor.transform(
    X_val_features
)

X_test = preprocessor.transform(
    X_test_features
)

In [44]:
from xgboost import XGBClassifier

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'aucpr'


In [45]:
import joblib

joblib.dump(
    preprocessor,
    "../data/processed/preprocessor.pkl"
)

joblib.dump(
    xgb_model,
    "../data/processed/fraud_model.pkl"
)

print("Model and preprocessor saved.")

Model and preprocessor saved.


In [46]:
loaded_preprocessor = joblib.load(
    "../data/processed/preprocessor.pkl"
)

loaded_model = joblib.load(
    "../data/processed/fraud_model.pkl"
)

X_test_loaded = loaded_preprocessor.transform(
    X_test_features
)

test_prob = loaded_model.predict_proba(
    X_test_loaded
)[:, 1]

print(test_prob[:10])

[0.9999999  0.99999905 0.99999976 0.9999989  0.9999999  0.9999987
 0.99999976 0.9999989  0.99999976 0.99999917]
